# Module 6.2 — Parent Document Retriever

**Dilemma**: small chunks → precise retrieval; large chunks → rich context for generation.

**Solution**: index *child* (small) chunks for retrieval, but return their *parent* (large) chunk to the LLM.

```
Parent chunk ──── split ──→ [child1] [child2] [child3] [child4]
                                 ↑ stored in vectorstore
Query matches child2 → return full parent chunk
```

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.storage import InMemoryStore
from langchain.retrievers import ParentDocumentRetriever
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

LONG_DOCS = [
    Document(page_content="""
Transformers are a type of neural network architecture introduced in the paper
'Attention Is All You Need' by Vaswani et al. in 2017. They rely entirely on
self-attention mechanisms to draw global dependencies between input and output,
dispensing with recurrence and convolutions entirely.

The transformer model consists of an encoder and a decoder, each composed of
multiple identical layers. Each layer contains two sub-layers: multi-head self-attention
and position-wise feed-forward networks. Layer normalisation and residual connections
are applied around each sub-layer.

The self-attention mechanism allows the model to weigh the importance of different
tokens in the input sequence when encoding each token. This is computed as:
Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V

Transformers have revolutionised NLP and are the foundation of models like BERT, GPT, T5, and LLaMA.
    """, metadata={"source": "transformers.txt"}),
]

# Child: 200 chars | Parent: full doc
child_splitter  = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=50)

embeddings  = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(collection_name="parent_child", embedding_function=embeddings)
docstore    = InMemoryStore()

retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

retriever.add_documents(LONG_DOCS, ids=None)

query = "What is the attention mechanism in transformers?"
child_docs = vectorstore.similarity_search(query, k=2)
parent_docs = retriever.invoke(query)

print(f"Child chunks matched: {len(child_docs)}")
for d in child_docs:
    print(f"  Child ({len(d.page_content)} chars): {d.page_content[:80].strip()}...")

print(f"\nParent docs returned: {len(parent_docs)}")
for d in parent_docs:
    print(f"  Parent ({len(d.page_content)} chars): {d.page_content[:120].strip()}...")
